# Stage 3 — Causal Feature Identification
**Owners: Person B + C** | Week 2

## Goal
Identify which SAE features are causally responsible for DINOv2 distinguishing
flamingo from spoonbill, using feature ablation.

## Person A implements first
Person A must have implemented src/causal.py run_ablation_loop() before this notebook runs.
Person B extends it to multiple layers. Person C runs the CaFE sanity check.

In [ ]:
import sys; sys.path.append("..")
from src.config import get_config
cfg = get_config()

## 1. Load behavior dataset (flamingo vs. spoonbill)

In [ ]:
from src.cache import load_layer, load_image_index, get_class_indices
from src.model import get_model, preprocess_batch, get_device

device = get_device()
model = get_model()

flamingo_idx  = get_class_indices(cfg.data.behavior_class_a)  # "flamingo"
spoonbill_idx = get_class_indices(cfg.data.behavior_class_b)  # "spoonbill"

print(f"Flamingo images:  {len(flamingo_idx)}")
print(f"Spoonbill images: {len(spoonbill_idx)}")

# Combine and load pixel values
all_idx = flamingo_idx[:cfg.data.behavior_n_images] + spoonbill_idx[:cfg.data.behavior_n_images]
image_index = load_image_index()
paths = image_index["paths"][all_idx].tolist()
pixel_values = preprocess_batch(paths).to(device)

## 2. Verify baseline logit difference

In [ ]:
from src.causal import compute_logit_diff

# Flamingo = ImageNet class 130, spoonbill = 129 (verify these!)
FLAMINGO_IDX  = 130
SPOONBILL_IDX = 129

logit_diffs = compute_logit_diff(model, pixel_values, FLAMINGO_IDX, SPOONBILL_IDX)
print(f"Mean logit diff: {logit_diffs.mean():.3f}")
# Positive = model leans toward flamingo on flamingo images (expected)

## 3. Run ablation loop — layer 11 (Person A)

In [ ]:
from src.cache import load_layer
from src.causal import run_ablation_loop, rank_features_by_importance

acts_11 = load_layer(layer=11, indices=all_idx)

effects_11 = run_ablation_loop(
    layer=11, activations=acts_11, model=model,
    pixel_values=pixel_values,
    class_a_idx=FLAMINGO_IDX, class_b_idx=SPOONBILL_IDX
)

important_11 = rank_features_by_importance(effects_11)
print(f"Important features at layer 11: {len(important_11)}")

## 4. Extend to layers 6 and 9 (Person B)

In [ ]:
from src.causal import run_ablation_all_layers

all_effects = run_ablation_all_layers(
    layers=cfg.sae.target_layers, model=model,
    pixel_values=pixel_values,
    class_a_idx=FLAMINGO_IDX, class_b_idx=SPOONBILL_IDX
)

# all_effects = {6: {...}, 9: {...}, 11: {...}}

## 5. CaFE sanity check (Person B)

In [ ]:
from src.causal import cafe_sanity_check
# top_patches_11 from notebook 02
# cafe_results = cafe_sanity_check(top_patches_11, model, pixel_values,
#                                   layer=11, important_features=important_11)
# from src.visualise import plot_cafe_comparison
# plot_cafe_comparison(cafe_results, top_patches_11, gradcam_maps, n_features=6)

## 6. Layer-wise evolution plot (Person C)

In [ ]:
# from src.visualise import plot_layer_evolution
# from src.evaluate import summarise_layer_evolution
# evolution = summarise_layer_evolution(ms_scores_per_layer, labels_per_layer)
# plot_layer_evolution(evolution, save=False)